# Day 2: Data Loading & Exploratory Data Analysis (`notebooks/01_eda.ipynb`)

This notebook implements the Day 2 EDA workflow using `src/data.py`:
1. Load `spectra`, `labels`, `wavelengths`, and `group_ids` via `src.data.load_spectra`
2. Run sanity checks (shape, dtypes, missing values, duplicates, wavelength axis range & spacing)
3. Inspect class balance & grouping structure (`plant_id`, `plot_id`, `field_id`, `date`)
4. Plot mean spectra per class with $\pm 1\text{ SD}$ variance bands
5. Plot difference spectrum, Fisher discriminability, Red Edge (690–750 nm), NIR plateau, outlier spectra, and temporal NDVI progression
6. Emit Day 3 band-drop recommendations and verify `docs/data_summary.md`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data import load_dataset, load_spectra

spectra, labels, wavelengths, group_ids = load_spectra('primary', drop_unlabeled=True)
print(f'Spectra shape: {spectra.shape} | dtype: {spectra.dtype}')
print(f'Wavelengths (nm): {wavelengths.tolist()}')
print(f'Spacings (nm): {np.diff(wavelengths).tolist()}')

In [ ]:
# Sanity checks: missing values, duplicates, unmatched metadata
ds = load_dataset('primary', drop_unlabeled=True)
print('Raw rows:', len(ds.raw_df), '| Unmatched metadata rows:', ds.unmatched_count)
print('Missing spectral values:', int(np.isnan(spectra).sum()))
print('Duplicate Sample_IDs:', int(group_ids['sample_id'].duplicated().sum()))
print('Exact duplicate spectral rows:', int(pd.DataFrame(spectra).duplicated().sum()))
print('\nClass balance:')
print(pd.Series(labels).value_counts())
print('\nUnique group counts:')
print(group_ids[['plant_id', 'plot_id', 'field_id', 'date']].nunique())

In [ ]:
# Run full automated Day 2 EDA & regenerate figures + docs/data_summary.md
from src.pipeline.run_day2_eda import main as run_eda
run_eda()

In [ ]:
# Display saved figures inline
from IPython.display import Image, display
for fig_name in [
    '01_mean_spectra_variance.png',
    '02_difference_spectrum_red_edge.png',
    '03_class_group_and_outliers.png',
    '04_temporal_spectral_progression.png',
]:
    display(Image(filename=str(ROOT / 'reports' / 'figures' / fig_name)))